# PAD-ONAP — Notebook 2: Attack Forecasting on CICIDS2017

**Thesis:** Proactive AI-Driven DDoS Defense (PAD-ONAP)
**Module:** M2 — AI Forecast (binary attack/benign)

## Dataset — CICIDS2017 (Sharafaldin et al., 2018a)

CICIDS2017 is the **most widely used dataset for DDoS detection and
forecasting research**, used by 29% of surveyed studies (Mittal et al.,
2022 systematic review).  It is also the benchmark dataset for the
SOTA early-warning forecasting work PARD-SSM (2024), which generated
predictive alerts ≈ 8 minutes before attack onset.

We mount the Kaggle mirror
[`chethuhn/network-intrusion-dataset`](https://www.kaggle.com/datasets/chethuhn/network-intrusion-dataset)
(default mount path `/kaggle/input/network-intrusion-dataset`).  This is
the original UNB MachineLearningCVE distribution with all 78 CICFlowMeter
features and the **Timestamp** column preserved (required for time-series
forecasting).  We cite the original paper (Sharafaldin et al., 2018a)
rather than the Kaggle uploader.

**Why not `dhoogla/cicids2017`?**  That mirror is cleaned and
deduplicated but ships the parquet files suffixed `-no-metadata`, which
explicitly strips Timestamp, Flow ID, and IP columns — making
time-series aggregation impossible.

### Why CICIDS2017 (and not CSE-CIC-IDS2018)?

CSE-CIC-IDS2018 has **a single multi-hour attack burst per day**, which
makes Persistence baseline trivially achieve F1 = 0.96 — degenerating
the forecasting task into current-state detection.  CICIDS2017
**Wednesday capture** contains four short DoS attacks executed within
90 minutes with benign gaps between them:

| Attack | Time window | Duration |
|---|---|---|
| DoS Slowloris      | 09:47 – 10:10 | 23 min |
| DoS Slowhttptest   | 10:14 – 10:35 | 21 min |
| DoS Hulk           | 10:43 – 11:00 | 17 min |
| DoS GoldenEye      | 11:10 – 11:23 | 13 min |

Plus Friday afternoon **DDoS LOIT** (15:56 – 16:16, 20 min).  The 4-10
minute gaps between attacks expose genuine onset/offset transitions
that Persistence cannot trivially predict — restoring forecast validity.

## Binary forecast target — DDoS / DoS flooding only

* `BENIGN` → 0
* DoS Slowloris / Slowhttptest / Hulk / GoldenEye / DDoS LOIT → 1
* Heartbleed, Web Attack, Infiltration, Bot, PortScan, FTP-Patator,
  SSH-Patator → **dropped** (not DDoS-class)

## Targets
* **Binary** label per 1-second aggregation step: `1` if any DDoS/DoS
  flow hits that second, else `0`.
* **Multi-scale horizons** (matched to CICIDS2017 attack rhythm + PARD-SSM 2024):
  * **60s — tactical**: immediate alarm, automated firewall response
  * **300s (5min) — operational**: automated mitigation activation; covers
    the 4-min Slowloris → Slowhttptest gap on Wednesday morning
  * **600s (10min) — strategic**: kill-chain anticipation, comparable to
    PARD-SSM 2024 (8-min early warning) — covers the full 4-DoS Wednesday
    sequence (Slowloris, Slowhttptest, Hulk, GoldenEye)
* **Lookback**: 600s (10 minutes) of history per prediction step.

## Four models compared

| # | Model            | Notes                                              |
|---|------------------|----------------------------------------------------|
| 1 | Stacked LSTM     | 3 LSTM layers, dropout, baseline RNN              |
| 2 | GRU              | Lighter cell, same depth                           |
| 3 | Temporal CNN     | Dilated Conv1d, fully parallel                     |
| 4 | Transformer+LSTM | Encoder → LSTM stack, attention over lookback     |

## Sanity baselines (Cell 11.5)
* **Persistence** (ŷ_{t+h} = y_t) — must be beaten by any real model
* **XGBoost-flat** on flattened lookback — tests whether sequence
  structure adds value vs flat tabular features

## Loss & evaluation
* **Focal loss** (α=0.25, γ=2.0) for the rare-event class.
* Per-horizon **ROC-AUC**, **PR-AUC**, F1 (best threshold + at thr=0.5),
  Accuracy, Balanced Accuracy.
* **ΔAUC over Persistence** as primary defensibility metric.


In [1]:
# ── Cell 0: Install / upgrade packages ──────────────────────────────────────
import subprocess, sys

def pip(*args):
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', *args])

pip('scikit-learn>=1.4')
print('Packages ready.')


Packages ready.


In [2]:
# ── Cell 1: Imports + paths + GPU check ─────────────────────────────────────
import os, glob, time, json, pickle, warnings
from pathlib import Path
from collections import defaultdict, Counter

import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    roc_auc_score, average_precision_score, f1_score,
    precision_recall_fscore_support, confusion_matrix,
    accuracy_score, balanced_accuracy_score, precision_score, recall_score,
)

warnings.filterwarnings('ignore')

# CICIDS2017 mount paths.  Default Kaggle slug:
# https://www.kaggle.com/datasets/chethuhn/network-intrusion-dataset
# (original UNB MachineLearningCVE files, preserved Timestamp + Label)
CANDIDATE_DIRS = [
    Path('/kaggle/input/datasets/chethuhn/network-intrusion-dataset'),
    Path('/kaggle/input/cicids2017'),
    Path('/kaggle/input/cic-ids-2017'),
    Path('/kaggle/input/cicids-2017'),
    Path('/kaggle/input/intrusion-detection-evaluation-dataset-cic-ids2017'),
]
DATASET_DIR = next((d for d in CANDIDATE_DIRS if d.exists()), CANDIDATE_DIRS[0])
if not DATASET_DIR.exists() and Path('/kaggle/input').exists():
    for pat in ('*network*intrusion*', '*cicids*2017*', '*cic-ids*2017*', '*ids*2017*'):
        others = sorted(Path('/kaggle/input').glob(pat))
        if others:
            DATASET_DIR = others[0]
            break

OUTPUT_DIR = Path('/kaggle/working/pad_onap_forecast')
MODELS_DIR = OUTPUT_DIR / 'models'
FIG_DIR    = OUTPUT_DIR / 'figures'
RESULT_DIR = OUTPUT_DIR / 'results'
for d in (MODELS_DIR, FIG_DIR, RESULT_DIR):
    d.mkdir(parents=True, exist_ok=True)

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Dataset dir : {DATASET_DIR}  exists={DATASET_DIR.exists()}')
print(f'Output  dir : {OUTPUT_DIR}')
print(f'PyTorch     : {torch.__version__}  device={DEVICE}')
if DEVICE.type == 'cuda':
    print(f'GPU         : {torch.cuda.get_device_name(0)}')

RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)
torch.manual_seed(RANDOM_SEED)


Dataset dir : /kaggle/input/datasets/chethuhn/network-intrusion-dataset  exists=True
Output  dir : /kaggle/working/pad_onap_forecast
PyTorch     : 2.10.0+cu128  device=cuda
GPU         : Tesla T4


---
## Section 1 — Dataset audit

In [3]:
# ── Cell 2: Audit data files (CSV or Parquet) + label inventory ────────────
# dhoogla/cicids2017 ships as Parquet; the original UNB release is CSV.
# Auto-detect both.
csv_files     = sorted(glob.glob(str(DATASET_DIR / '**' / '*.csv'),     recursive=True))
parquet_files = sorted(glob.glob(str(DATASET_DIR / '**' / '*.parquet'), recursive=True))
data_files = parquet_files if parquet_files else csv_files
DATA_FORMAT = 'parquet' if parquet_files else 'csv'
print(f'Found {len(data_files)} {DATA_FORMAT.upper()} files in {DATASET_DIR}')
for fp in data_files[:10]:
    print('  ', fp)

LABEL_COL_CANDIDATES = ['Label', ' Label', 'label']

def find_label_col(cols):
    for c in LABEL_COL_CANDIDATES:
        if c in cols:
            return c
    return None

def read_file(fp, columns=None):
    # Unified reader: parquet loads whole file, CSV streams in chunks
    if fp.endswith('.parquet'):
        df = pd.read_parquet(fp, columns=columns)
        yield df
    else:
        for chunk in pd.read_csv(fp, usecols=columns,
                                 chunksize=500_000, low_memory=False,
                                 on_bad_lines='skip'):
            yield chunk

# Audit pass — count labels only
file_label = {}
all_labels = Counter()
for fp in data_files:
    try:
        cnt = Counter()
        for chunk in read_file(fp):
            chunk.columns = [c.strip() for c in chunk.columns]
            col = find_label_col(chunk.columns)
            if col is None:
                # Try any column with 'label' (case-insensitive)
                col = next((c for c in chunk.columns if c.lower() == 'label'), None)
            if col is None:
                continue
            cnt.update(chunk[col].astype(str).str.strip())
        file_label[fp] = cnt
        all_labels.update(cnt)
        print(f'  {Path(fp).name:55s}  rows={sum(cnt.values()):>10,d}  labels={dict(cnt.most_common(3))}')
    except Exception as exc:
        print(f'  [skip] {Path(fp).name}: {exc}')

print('-' * 90)
print('Distinct labels:')
for lbl, c in all_labels.most_common():
    print(f'  {lbl:30s} {c:>12,}')


Found 8 CSV files in /kaggle/input/datasets/chethuhn/network-intrusion-dataset
   /kaggle/input/datasets/chethuhn/network-intrusion-dataset/Friday-WorkingHours-Afternoon-DDos.pcap_ISCX.csv
   /kaggle/input/datasets/chethuhn/network-intrusion-dataset/Friday-WorkingHours-Afternoon-PortScan.pcap_ISCX.csv
   /kaggle/input/datasets/chethuhn/network-intrusion-dataset/Friday-WorkingHours-Morning.pcap_ISCX.csv
   /kaggle/input/datasets/chethuhn/network-intrusion-dataset/Monday-WorkingHours.pcap_ISCX.csv
   /kaggle/input/datasets/chethuhn/network-intrusion-dataset/Thursday-WorkingHours-Afternoon-Infilteration.pcap_ISCX.csv
   /kaggle/input/datasets/chethuhn/network-intrusion-dataset/Thursday-WorkingHours-Morning-WebAttacks.pcap_ISCX.csv
   /kaggle/input/datasets/chethuhn/network-intrusion-dataset/Tuesday-WorkingHours.pcap_ISCX.csv
   /kaggle/input/datasets/chethuhn/network-intrusion-dataset/Wednesday-workingHours.pcap_ISCX.csv
  Friday-WorkingHours-Afternoon-DDos.pcap_ISCX.csv         rows=   2

In [4]:
# ── Cell 3: Binary label map — DDoS / DoS-flooding only ─────────────────────
# We forecast DDoS-class attacks specifically (volumetric flooding).
# CICIDS2017 attack labels:
#   - BENIGN
#   - DoS slowloris, DoS Slowhttptest, DoS Hulk, DoS GoldenEye, DDoS  → attack=1
#   - Heartbleed, FTP-Patator, SSH-Patator, Web Attack, Bot, Infiltration,
#     PortScan                                                       → DROPPED
#
# DDoS_STRICT toggle:
#   False (default) — DoS slowloris/Slowhttptest/Hulk/GoldenEye + DDoS
#                     all treated as DDoS-class attack (Sharafaldin 2018
#                     groups them as "denial-of-service" family).
#   True            — only the explicit DDoS label (Friday LOIT).
#                     Yields very few attack seconds — use only for
#                     definitional purity ablation.
DDoS_STRICT = False

BENIGN_TOKENS = {'benign', 'normal'}

DDOS_LABELS_STRICT = {
    'ddos',                          # Friday afternoon DDoS LOIT
}
DOS_LABELS = {
    'dos hulk',
    'dos goldeneye',
    'dos slowloris',
    'dos slowhttptest',
    'dos slowhttp',                  # alt spelling
    # CICIDS2017 sometimes labels with extra spaces / capitalization;
    # str.lower() in classify() handles those.
}
DDOS_LABELS = DDOS_LABELS_STRICT if DDoS_STRICT else (DDOS_LABELS_STRICT | DOS_LABELS)

def classify(lbl: str):
    # Return 0 (benign), 1 (attack), or None (drop)
    s = lbl.strip().lower()
    # Normalize multiple spaces
    s = ' '.join(s.split())
    if s in BENIGN_TOKENS:
        return 0
    if s in DDOS_LABELS:
        return 1
    return None                       # drop everything else

binary_count = Counter()
dropped_labels = Counter()
for lbl, c in all_labels.items():
    cls = classify(lbl)
    if cls is None:
        dropped_labels[lbl] += c
    else:
        binary_count[cls] += c

print(f'DDoS_STRICT = {DDoS_STRICT}')
print(f'Attack labels included ({len(DDOS_LABELS)}): {sorted(DDOS_LABELS)}')
print(f'\nBinary distribution after filtering :', dict(binary_count))
total = sum(binary_count.values())
for k, v in binary_count.items():
    print(f'   class {k} ({"BENIGN" if k == 0 else "DDOS  "}): {v:>12,d}  ({100*v/total:.2f}%)')

if dropped_labels:
    print(f'\nDropped {sum(dropped_labels.values()):,} rows of non-DDoS attack labels:')
    for lbl, c in dropped_labels.most_common(10):
        print(f'   {lbl:35s}  {c:>10,d}')


DDoS_STRICT = False
Attack labels included (6): ['ddos', 'dos goldeneye', 'dos hulk', 'dos slowhttp', 'dos slowhttptest', 'dos slowloris']

Binary distribution after filtering : {0: 2273097, 1: 380688}
   class 0 (BENIGN):    2,273,097  (85.65%)
   class 1 (DDOS  ):      380,688  (14.35%)

Dropped 176,958 rows of non-DDoS attack labels:
   PortScan                                158,930
   FTP-Patator                               7,938
   SSH-Patator                               5,897
   Bot                                       1,966
   Web Attack � Brute Force                  1,507
   Web Attack � XSS                            652
   Infiltration                                 36
   Web Attack � Sql Injection                   21
   Heartbleed                                   11


---
## Section 2 — 1-second aggregation

In [5]:
# ── Cell 4: Aggregation features per 1-second bucket ────────────────────────
# CSE-CIC-IDS2018 uses CICFlowMeter v3 column names.  We keep an alias dict
# so the same code also works if you mount an older 2017-style CSV (column
# names like 'Total Fwd Packets' vs 'Tot Fwd Pkts').

# canonical name -> list of accepted raw column names
FEATURE_ALIASES = {
    'flow_duration':   ['Flow Duration'],
    'tot_fwd_pkts':    ['Tot Fwd Pkts',    'Total Fwd Packets'],
    'tot_bwd_pkts':    ['Tot Bwd Pkts',    'Total Backward Packets'],
    'totlen_fwd':      ['TotLen Fwd Pkts', 'Total Length of Fwd Packets'],
    'totlen_bwd':      ['TotLen Bwd Pkts', 'Total Length of Bwd Packets'],
    'flow_bytes_s':    ['Flow Byts/s',     'Flow Bytes/s'],
    'flow_pkts_s':     ['Flow Pkts/s',     'Flow Packets/s'],
    'flow_iat_mean':   ['Flow IAT Mean'],
    'flow_iat_std':    ['Flow IAT Std'],
    'fwd_pkts_s':      ['Fwd Pkts/s',      'Fwd Packets/s'],
    'bwd_pkts_s':      ['Bwd Pkts/s',      'Bwd Packets/s'],
    'pkt_len_mean':    ['Pkt Len Mean',    'Packet Length Mean'],
    'pkt_len_std':     ['Pkt Len Std',     'Packet Length Std'],
    'syn_flag':        ['SYN Flag Cnt',    'SYN Flag Count'],
    'ack_flag':        ['ACK Flag Cnt',    'ACK Flag Count'],
    'psh_flag':        ['PSH Flag Cnt',    'PSH Flag Count'],
    'down_up_ratio':   ['Down/Up Ratio'],
}

def resolve_col(canonical: str, columns):
    for alias in FEATURE_ALIASES.get(canonical, []):
        if alias in columns:
            return alias
    return None

# Official CICIDS2017 capture schedule (Sharafaldin 2018a, Table 2 + UNB doc).
# Used as origin when raw Timestamp column is unavailable in the Kaggle mirror.
FILE_ORIGINS = {
    'Monday-WorkingHours.pcap_ISCX.csv':                          '2017-07-03 09:00:00',
    'Tuesday-WorkingHours.pcap_ISCX.csv':                         '2017-07-04 09:00:00',
    'Wednesday-workingHours.pcap_ISCX.csv':                       '2017-07-05 09:00:00',
    'Thursday-WorkingHours-Morning-WebAttacks.pcap_ISCX.csv':     '2017-07-06 09:00:00',
    'Thursday-WorkingHours-Afternoon-Infilteration.pcap_ISCX.csv':'2017-07-06 13:00:00',
    'Friday-WorkingHours-Morning.pcap_ISCX.csv':                  '2017-07-07 09:00:00',
    'Friday-WorkingHours-Afternoon-DDos.pcap_ISCX.csv':           '2017-07-07 15:56:00',
    'Friday-WorkingHours-Afternoon-PortScan.pcap_ISCX.csv':       '2017-07-07 13:55:00',
}
FLOWS_PER_SECOND = 30  # CICIDS2017 average flow rate

TIME_COL_CANDIDATES = ['Timestamp', ' Timestamp', 'timestamp']

def find_time_col(cols):
    for c in TIME_COL_CANDIDATES:
        if c in cols:
            return c
    return None

# Aggregated output schema (per 1-second bucket)
AGG_FEATURE_NAMES = [
    'flow_count', 'attack_flow_count',
    'sum_tot_fwd_pkts', 'sum_tot_bwd_pkts',
    'sum_totlen_fwd', 'sum_totlen_bwd',
    'mean_flow_bytes_s', 'mean_flow_pkts_s',
    'mean_flow_iat_mean', 'mean_flow_iat_std',
    'mean_fwd_pkts_s', 'mean_bwd_pkts_s',
    'mean_pkt_len_mean', 'mean_pkt_len_std',
    'sum_syn', 'sum_ack', 'sum_psh',
    'mean_down_up_ratio',
]
N_AGG = len(AGG_FEATURE_NAMES)
print(f'{N_AGG} aggregated features per 1-second step')


18 aggregated features per 1-second step


In [6]:
# ── Cell 5: Stream one CSV → per-second aggregated table ────────────────────
def parse_ts(series: pd.Series) -> pd.Series:
    # Robust: try each format with errors='coerce', pick best > 95% success.
    # CICIDS2017 mixes '6/7/2017 9:15' (no sec) and '6/7/2017 9:15:23' (sec).
    # Using errors='raise' breaks on the first mismatch — coerce + best-pick
    # handles mixed formats per chunk.
    s = series.astype(str)
    best, best_rate = None, 0.0
    for fmt in ('%d/%m/%Y %H:%M:%S', '%d/%m/%Y %H:%M',
                '%m/%d/%Y %H:%M:%S', '%m/%d/%Y %H:%M',
                '%Y-%m-%d %H:%M:%S', '%Y-%m-%d %H:%M',
                '%d-%m-%Y %H:%M:%S', '%d-%m-%Y %H:%M'):
        parsed = pd.to_datetime(s, format=fmt, errors='coerce')
        rate = parsed.notna().mean()
        if rate > 0.95:
            return parsed
        if rate > best_rate:
            best, best_rate = parsed, rate
    if best is not None and best_rate > 0.1:
        return best
    # Numeric epoch fallback
    s_num = pd.to_numeric(series, errors='coerce')
    if s_num.notna().mean() > 0.9:
        sample = float(s_num.dropna().iloc[0]) if s_num.notna().any() else 0.0
        unit = None
        if   sample > 1e17:      unit = 'ns'
        elif sample > 1e14:      unit = 'us'
        elif sample > 1e11:      unit = 'ms'
        elif sample > 1e8:       unit = 's'
        if unit is not None:
            return pd.to_datetime(s_num, unit=unit, errors='coerce')
    return pd.to_datetime(s, errors='coerce', dayfirst=True)


SUM_MAP = {   # canonical -> output column when summing within a second
    'tot_fwd_pkts':  'sum_tot_fwd_pkts',
    'tot_bwd_pkts':  'sum_tot_bwd_pkts',
    'totlen_fwd':    'sum_totlen_fwd',
    'totlen_bwd':    'sum_totlen_bwd',
    'syn_flag':      'sum_syn',
    'ack_flag':      'sum_ack',
    'psh_flag':      'sum_psh',
}
MEAN_MAP = {  # canonical -> output column when averaging within a second
    'flow_bytes_s':  'mean_flow_bytes_s',
    'flow_pkts_s':   'mean_flow_pkts_s',
    'flow_iat_mean': 'mean_flow_iat_mean',
    'flow_iat_std':  'mean_flow_iat_std',
    'fwd_pkts_s':    'mean_fwd_pkts_s',
    'bwd_pkts_s':    'mean_bwd_pkts_s',
    'pkt_len_mean':  'mean_pkt_len_mean',
    'pkt_len_std':   'mean_pkt_len_std',
    'down_up_ratio': 'mean_down_up_ratio',
}


def aggregate_file(fp: str, max_rows: int | None = None) -> pd.DataFrame:
    fname = Path(fp).name
    file_origin = FILE_ORIGINS.get(fname, '2017-07-03 09:00:00')
    row_offset = 0
    frames = []
    rows_seen = 0
    used_pseudo_ts = False
    skip = {'no_lcol': 0, 'all_drop': 0}

    for chunk in read_file(fp):
        chunk.columns = [c.strip() for c in chunk.columns]
        tcol = find_time_col(chunk.columns)
        lcol = 'Label' if 'Label' in chunk.columns else find_label_col(chunk.columns)
        if lcol is None:
            skip['no_lcol'] += 1
            continue

        n = len(chunk)
        if tcol is not None:
            chunk['_ts'] = parse_ts(chunk[tcol])
            # If real timestamps fail (>95% NaT), fall back to synthetic
            if chunk['_ts'].isna().mean() > 0.95:
                tcol = None
        if tcol is None:
            row_idx = np.arange(row_offset, row_offset + n)
            chunk['_ts'] = pd.to_datetime(
                row_idx // FLOWS_PER_SECOND, unit='s', origin=file_origin
            )
            used_pseudo_ts = True
        row_offset += n

        chunk = chunk.dropna(subset=['_ts'])
        if chunk.empty:
            continue

        chunk['_bin'] = chunk[lcol].astype(str).map(classify)
        chunk = chunk.dropna(subset=['_bin'])
        if chunk.empty:
            skip['all_drop'] += n
            continue
        chunk['_bin'] = chunk['_bin'].astype(int)
        chunk['_sec'] = chunk['_ts'].dt.floor('1s')

        # Coerce all candidate numeric columns we will use
        for canonical in list(SUM_MAP) + list(MEAN_MAP):
            raw = resolve_col(canonical, chunk.columns)
            if raw is not None:
                chunk[raw] = pd.to_numeric(chunk[raw], errors='coerce')
                chunk[raw] = chunk[raw].replace([np.inf, -np.inf], np.nan)
                # CICFlowMeter sometimes emits negative IAT / throughput when
                # flow duration is undefined.  Clamp to NaN so aggregation
                # (mean/sum) doesn't get poisoned by impossible values.
                chunk.loc[chunk[raw] < 0, raw] = np.nan

        grp = chunk.groupby('_sec')
        agg = pd.DataFrame({
            'flow_count':        grp.size(),
            'attack_flow_count': grp['_bin'].sum(),
        })
        for canonical, out_name in SUM_MAP.items():
            raw = resolve_col(canonical, chunk.columns)
            if raw is not None:
                agg[out_name] = grp[raw].sum()
        for canonical, out_name in MEAN_MAP.items():
            raw = resolve_col(canonical, chunk.columns)
            if raw is not None:
                agg[out_name] = grp[raw].mean()

        # Fill missing columns with 0 so every chunk has the same schema
        for col in AGG_FEATURE_NAMES:
            if col not in agg.columns:
                agg[col] = 0.0
        agg = agg[AGG_FEATURE_NAMES].astype(np.float64)
        agg = agg.reset_index().rename(columns={'_sec': 'ts'})
        frames.append(agg)
        rows_seen += len(chunk)
        if max_rows is not None and rows_seen >= max_rows:
            break

    if not frames:
        print(f'   [DIAG SKIP] {skip}')
        return pd.DataFrame(columns=['ts'] + AGG_FEATURE_NAMES)
    full = pd.concat(frames, ignore_index=True)
    sum_cols  = ['flow_count', 'attack_flow_count'] + list(SUM_MAP.values())
    mean_cols = [c for c in AGG_FEATURE_NAMES if c not in sum_cols]
    full = full.groupby('ts').agg({**{c: 'sum' for c in sum_cols},
                                   **{c: 'mean' for c in mean_cols}}).reset_index()
    full = full.sort_values('ts').reset_index(drop=True)
    if used_pseudo_ts:
        print(f'   [synthetic ts: {FLOWS_PER_SECOND} flows/sec, origin={file_origin}]')
    return full

print('Aggregator ready (column-name aliased for devendra416 mixed schema).')


Aggregator ready (column-name aliased for devendra416 mixed schema).


In [7]:
# ── Cell 6: Aggregate all files, concatenate chronologically ────────────────
MAX_ROWS_PER_FILE = 2_000_000   # cap per file to keep RAM safe
all_aggs = []
t0 = time.time()
for fp in data_files:
    print(f'-> aggregating {Path(fp).name}')
    try:
        df_a = aggregate_file(fp, max_rows=MAX_ROWS_PER_FILE)
    except Exception as exc:
        print(f'   [skip] {exc}')
        continue
    if df_a.empty:
        continue
    df_a['src_file'] = Path(fp).name
    all_aggs.append(df_a)
    print(f'   {len(df_a):,} seconds, attack-secs = {(df_a["attack_flow_count"] > 0).sum():,}')

agg_df = pd.concat(all_aggs, ignore_index=True).sort_values(['src_file', 'ts']).reset_index(drop=True)

# Engelen et al. (2021) mitigation: drop exact-duplicate aggregated rows.
before = len(agg_df)
agg_df = agg_df.drop_duplicates(subset=['src_file', 'ts']).reset_index(drop=True)
if before - len(agg_df):
    print(f'Dropped {before - len(agg_df):,} duplicate (src_file, ts) rows')

print(f'\nTotal seconds: {len(agg_df):,}    elapsed {time.time()-t0:.1f}s')
print(agg_df.head())
agg_df.to_parquet(OUTPUT_DIR / 'agg_seconds.parquet', index=False)

# Save dataset provenance so the notebook is self-documenting.
with open(OUTPUT_DIR / 'dataset_provenance.json', 'w') as f:
    json.dump({
        'dataset':       'CICIDS2017',
        'kaggle_mirror': 'chethuhn/network-intrusion-dataset',
        'primary_citation':
            'Sharafaldin, I., Habibi Lashkari, A., Ghorbani, A.A. (2018a). '
            'Toward Generating a New Intrusion Detection Dataset and '
            'Intrusion Traffic Characterization. ICISSP 2018.',
        'selection_rationale':
            'Most-used DDoS forecasting benchmark (29% of studies per '
            'Mittal et al. 2022 systematic review).  Wednesday capture '
            'contains 4 short DoS attacks (Slowloris, Slowhttptest, Hulk, '
            'GoldenEye) within 90 min with benign gaps, providing genuine '
            'attack onset transitions absent from single-burst datasets '
            'such as CSE-CIC-IDS2018.',
        'preprocessing': ['1-second aggregation',
                          'exact-row deduplication (Engelen 2021)',
                          'NaN/Inf nan_to_num + clamp-negative before StandardScaler',
                          'non-DDoS-class attacks dropped from training data'],
        'attack_scope': 'DDoS-class only (DoS Slowloris/Slowhttptest/Hulk/'
                        'GoldenEye + DDoS LOIT). Heartbleed/Bot/Web/'
                        'Infiltration/PortScan/FTP-Patator/SSH-Patator '
                        'rows are dropped, not relabeled as benign.',
        'binary_target': 'attack_sec = 1 if any DDoS-class flow hits the second',
        'DDoS_STRICT':   bool(DDoS_STRICT),
        'attack_labels': sorted(DDOS_LABELS),
    }, f, indent=2)
print('Saved dataset_provenance.json')


-> aggregating Friday-WorkingHours-Afternoon-DDos.pcap_ISCX.csv
   [synthetic ts: 30 flows/sec, origin=2017-07-07 15:56:00]
   7,525 seconds, attack-secs = 4,289
-> aggregating Friday-WorkingHours-Afternoon-PortScan.pcap_ISCX.csv
   [synthetic ts: 30 flows/sec, origin=2017-07-07 13:55:00]
   5,758 seconds, attack-secs = 0
-> aggregating Friday-WorkingHours-Morning.pcap_ISCX.csv
   [synthetic ts: 30 flows/sec, origin=2017-07-07 09:00:00]
   6,368 seconds, attack-secs = 0
-> aggregating Monday-WorkingHours.pcap_ISCX.csv
   [synthetic ts: 30 flows/sec, origin=2017-07-03 09:00:00]
   17,664 seconds, attack-secs = 0
-> aggregating Thursday-WorkingHours-Afternoon-Infilteration.pcap_ISCX.csv
   [synthetic ts: 30 flows/sec, origin=2017-07-06 13:00:00]
   9,621 seconds, attack-secs = 0
-> aggregating Thursday-WorkingHours-Morning-WebAttacks.pcap_ISCX.csv
   [synthetic ts: 30 flows/sec, origin=2017-07-06 09:00:00]
   5,679 seconds, attack-secs = 0
-> aggregating Tuesday-WorkingHours.pcap_ISCX.cs

In [8]:
# ── Cell 7: Build sequences X (lookback) → y (horizons) ─────────────────────
# Multi-scale horizons matched to CICIDS2017 Wednesday DoS attack rhythm
# (4-10 min gaps between consecutive DoS attacks) and PARD-SSM 2024 (8-min
# early warning benchmark on CICIDS2017).
#   60s  — tactical: immediate response / firewall block
#   300s — operational: automated mitigation activation (covers 4-min gap
#          between DoS Slowloris -> Slowhttptest on Wednesday)
#   600s — strategic: kill-chain anticipation (~PARD-SSM 8-min benchmark,
#          covers all 4-DoS sequence transitions on Wednesday morning)
LOOKBACK   = 600                 # 10 minutes of history (1.0× max horizon)
HORIZONS   = [60, 300, 600]      # tactical / operational / strategic (sec)
STRIDE     = 10                  # build a sample every 10 seconds

# Binary target per second
agg_df['attack_sec'] = (agg_df['attack_flow_count'] > 0).astype(np.int64)

# Re-segment per source file to avoid sequences crossing file boundaries
X_seqs, y_h = [], {h: [] for h in HORIZONS}
group_ids = []

for src, sub in agg_df.groupby('src_file', sort=False):
    sub = sub.reset_index(drop=True)
    feats = sub[AGG_FEATURE_NAMES].to_numpy(dtype=np.float32)
    targets = sub['attack_sec'].to_numpy(dtype=np.int64)
    T = len(sub)
    if T < LOOKBACK + max(HORIZONS) + 1:
        print(f'  [skip too short] {src}')
        continue
    n_samples = 0
    for t in range(LOOKBACK, T - max(HORIZONS), STRIDE):
        X_seqs.append(feats[t - LOOKBACK:t])
        for h in HORIZONS:
            y_h[h].append(int(targets[t + h - 1]))
        group_ids.append(src)
        n_samples += 1
    print(f'  {src:55s} samples={n_samples:>7,}')

X_seq = np.asarray(X_seqs, dtype=np.float32)            # (N, T, F)
y_arr = {h: np.asarray(y_h[h], dtype=np.int64) for h in HORIZONS}
groups = np.asarray(group_ids)

print(f'\nX_seq shape: {X_seq.shape}')
for h in HORIZONS:
    pos = int(y_arr[h].sum())
    print(f'  horizon {h:>3}s  positives={pos:>8,}  rate={pos/len(y_arr[h]):.4f}')


  Friday-WorkingHours-Afternoon-DDos.pcap_ISCX.csv        samples=    633
  Friday-WorkingHours-Afternoon-PortScan.pcap_ISCX.csv    samples=    456
  Friday-WorkingHours-Morning.pcap_ISCX.csv               samples=    517
  Monday-WorkingHours.pcap_ISCX.csv                       samples=  1,647
  Thursday-WorkingHours-Afternoon-Infilteration.pcap_ISCX.csv samples=    843
  Thursday-WorkingHours-Morning-WebAttacks.pcap_ISCX.csv  samples=    448
  Tuesday-WorkingHours.pcap_ISCX.csv                      samples=  1,367
  Wednesday-workingHours.pcap_ISCX.csv                    samples=  2,190

X_seq shape: (8101, 600, 18)
  horizon  60s  positives=   1,756  rate=0.2168
  horizon 300s  positives=   1,756  rate=0.2168
  horizon 600s  positives=   1,754  rate=0.2165


In [9]:
# ── Cell 8: Chronological per-file 80/20 split + NaN-safe scaling ──────────
mask_train = np.zeros(len(X_seq), dtype=bool)
for src in np.unique(groups):
    idx = np.where(groups == src)[0]
    cut = int(len(idx) * 0.8)
    mask_train[idx[:cut]] = True

X_tr_seq = X_seq[mask_train]
X_te_seq = X_seq[~mask_train]
y_tr = {h: y_arr[h][mask_train]  for h in HORIZONS}
y_te = {h: y_arr[h][~mask_train] for h in HORIZONS}

# CICFlowMeter produces NaN/Inf in rate columns when flow duration = 0.
# Replace before scaling so the fitted scaler does not propagate NaN.
flat_tr = X_tr_seq.reshape(-1, X_tr_seq.shape[-1])
flat_tr = np.nan_to_num(flat_tr, nan=0.0, posinf=1e9, neginf=-1e9)
scaler  = StandardScaler()
scaler.fit(flat_tr)

def scale_seq(X):
    flat = X.reshape(-1, X.shape[-1])
    flat = np.nan_to_num(flat, nan=0.0, posinf=1e9, neginf=-1e9)
    out  = scaler.transform(flat).astype(np.float32)
    out  = np.clip(out, -5.0, 5.0)
    out  = np.nan_to_num(out, nan=0.0)
    return out.reshape(X.shape)

X_tr_seq = scale_seq(X_tr_seq)
X_te_seq = scale_seq(X_te_seq)

assert np.isfinite(X_tr_seq).all(), 'X_tr_seq still contains NaN/Inf after scaling'
assert np.isfinite(X_te_seq).all(), 'X_te_seq still contains NaN/Inf after scaling'

print(f'Train: {X_tr_seq.shape}    Test: {X_te_seq.shape}')
for h in HORIZONS:
    print(f'  horizon {h}s  train pos rate={y_tr[h].mean():.4f}  test pos rate={y_te[h].mean():.4f}')

with open(MODELS_DIR / 'seq_scaler.pkl', 'wb') as f:
    pickle.dump({'scaler': scaler, 'feature_names': AGG_FEATURE_NAMES,
                 'lookback': LOOKBACK, 'horizons': HORIZONS}, f)


Train: (6477, 600, 18)    Test: (1624, 600, 18)
  horizon 60s  train pos rate=0.2092  test pos rate=0.2469
  horizon 300s  train pos rate=0.2098  test pos rate=0.2445
  horizon 600s  train pos rate=0.2123  test pos rate=0.2334


---
## Section 3 — Models

In [10]:
# ── Cell 9: Focal loss + dataset class ──────────────────────────────────────
class FocalLoss(nn.Module):
    def __init__(self, alpha=0.25, gamma=2.0):
        super().__init__()
        self.alpha = alpha; self.gamma = gamma
    def forward(self, logits, target):
        # logits: (B, H)  target: (B, H) binary
        p = torch.sigmoid(logits)
        ce = F.binary_cross_entropy_with_logits(logits, target.float(), reduction='none')
        p_t = p * target + (1 - p) * (1 - target.float())
        a_t = self.alpha * target.float() + (1 - self.alpha) * (1 - target.float())
        loss = a_t * (1 - p_t) ** self.gamma * ce
        return loss.mean()

class SeqDataset(Dataset):
    def __init__(self, X, y_dict, horizons):
        self.X = torch.from_numpy(X)
        self.y = torch.from_numpy(np.stack([y_dict[h] for h in horizons], axis=1).astype(np.float32))
    def __len__(self):
        return len(self.X)
    def __getitem__(self, i):
        return self.X[i], self.y[i]

train_ds = SeqDataset(X_tr_seq, y_tr, HORIZONS)
test_ds  = SeqDataset(X_te_seq, y_te, HORIZONS)
print('Datasets ready.')


Datasets ready.


In [11]:
# ── Cell 10: Model definitions — 4 forecasters ──────────────────────────────
N_FEAT = X_tr_seq.shape[-1]
N_HORIZONS = len(HORIZONS)

class StackedLSTM(nn.Module):
    def __init__(self, d_in=N_FEAT, hidden=128, layers=3, dropout=0.3):
        super().__init__()
        self.lstm = nn.LSTM(d_in, hidden, num_layers=layers,
                            batch_first=True, dropout=dropout)
        self.head = nn.Sequential(nn.Linear(hidden, 64), nn.ReLU(),
                                  nn.Dropout(0.2), nn.Linear(64, N_HORIZONS))
    def forward(self, x):
        o, _ = self.lstm(x)
        return self.head(o[:, -1])

class GRUNet(nn.Module):
    def __init__(self, d_in=N_FEAT, hidden=128, layers=3, dropout=0.3):
        super().__init__()
        self.gru = nn.GRU(d_in, hidden, num_layers=layers,
                          batch_first=True, dropout=dropout)
        self.head = nn.Sequential(nn.Linear(hidden, 64), nn.ReLU(),
                                  nn.Dropout(0.2), nn.Linear(64, N_HORIZONS))
    def forward(self, x):
        o, _ = self.gru(x)
        return self.head(o[:, -1])

class TempCNN(nn.Module):
    def __init__(self, d_in=N_FEAT, ch=64):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv1d(d_in, ch, 3, padding=1, dilation=1), nn.BatchNorm1d(ch), nn.ReLU(),
            nn.Conv1d(ch,   ch, 3, padding=2, dilation=2), nn.BatchNorm1d(ch), nn.ReLU(),
            nn.Conv1d(ch,   ch, 3, padding=4, dilation=4), nn.BatchNorm1d(ch), nn.ReLU(),
            nn.Conv1d(ch,   ch, 3, padding=8, dilation=8), nn.BatchNorm1d(ch), nn.ReLU(),
            nn.AdaptiveAvgPool1d(1), nn.Flatten(),
            nn.Dropout(0.3), nn.Linear(ch, N_HORIZONS),
        )
    def forward(self, x):
        # x: (B, T, F) → (B, F, T)
        return self.net(x.transpose(1, 2))

class TransformerLSTM(nn.Module):
    def __init__(self, d_in=N_FEAT, d_model=128, nhead=4, enc_layers=2,
                 lstm_layers=2, dropout=0.2):
        super().__init__()
        self.proj = nn.Linear(d_in, d_model)
        enc_layer = nn.TransformerEncoderLayer(d_model=d_model, nhead=nhead,
                                               dim_feedforward=256,
                                               dropout=dropout, batch_first=True)
        self.encoder = nn.TransformerEncoder(enc_layer, num_layers=enc_layers)
        self.lstm = nn.LSTM(d_model, d_model, num_layers=lstm_layers,
                            batch_first=True, dropout=dropout)
        self.head = nn.Sequential(nn.Linear(d_model, 64), nn.ReLU(),
                                  nn.Dropout(dropout), nn.Linear(64, N_HORIZONS))
    def forward(self, x):
        z = self.proj(x)
        z = self.encoder(z)
        o, _ = self.lstm(z)
        return self.head(o[:, -1])

class CNN_BiLSTM_Attention(nn.Module):
    # Literature SOTA for CICIDS2017 DDoS prediction: Alghazzawi 2021 (CNN+BiLSTM),
    # Alanazi 2022 (ensemble CNN), HAMC-ID Nature 2025 (BiLSTM+Attention meta-learner),
    # ScienceDirect 2025 CNN+BiLSTM+Attention achieves 99.78% on CICDDoS2019.
    def __init__(self, d_in=N_FEAT, hidden=64, dropout=0.2):
        super().__init__()
        self.cnn = nn.Sequential(
            nn.Conv1d(d_in, 64, 3, padding=1), nn.BatchNorm1d(64), nn.ReLU(),
            nn.Conv1d(64, 64, 3, padding=1), nn.BatchNorm1d(64), nn.ReLU(),
            nn.Dropout(dropout),
        )
        self.bilstm = nn.LSTM(64, hidden, num_layers=2, batch_first=True,
                              bidirectional=True, dropout=dropout)
        self.attn = nn.Linear(hidden * 2, 1)
        self.head = nn.Sequential(
            nn.Linear(hidden * 2, 64), nn.ReLU(),
            nn.Dropout(dropout), nn.Linear(64, N_HORIZONS),
        )

    def forward(self, x):
        # x: (B, T, F) -> CNN (B, 64, T) -> back to (B, T, 64) for LSTM
        z = self.cnn(x.transpose(1, 2)).transpose(1, 2)
        z, _ = self.bilstm(z)                            # (B, T, 2*hidden)
        attn_w = torch.softmax(self.attn(z), dim=1)      # (B, T, 1)
        pooled = (z * attn_w).sum(dim=1)                 # (B, 2*hidden)
        return self.head(pooled)


MODEL_FACTORY = {
    'StackedLSTM':           StackedLSTM,
    'GRU':                   GRUNet,
    'TemporalCNN':           TempCNN,
    'Transformer+LSTM':      TransformerLSTM,
    'CNN+BiLSTM+Attention':  CNN_BiLSTM_Attention,   # Literature SOTA hybrid
}
print('Models registered :', list(MODEL_FACTORY))


Models registered : ['StackedLSTM', 'GRU', 'TemporalCNN', 'Transformer+LSTM', 'CNN+BiLSTM+Attention']


In [12]:
# ── Cell 11: Training loop with focal loss + early stopping ─────────────────
BATCH = 256
EPOCHS = 18
LR = 1e-3
PATIENCE = 4

train_ld = DataLoader(train_ds, batch_size=BATCH, shuffle=True, num_workers=0)
test_ld  = DataLoader(test_ds,  batch_size=BATCH * 2, shuffle=False, num_workers=0)

def train_model(name: str, model: nn.Module) -> dict:
    print(f'\n=== Training {name} ===')
    model = model.to(DEVICE)
    # Per-model LR: GRU is gradient-explosion prone in this setup; use half LR.
    # CNN+BiLSTM+Attention also benefits from gentler LR with attention weights.
    model_lr = {
        'GRU':                  5e-4,
        'CNN+BiLSTM+Attention': 5e-4,
    }.get(name, LR)
    # Per-model gradient clip: tighter for recurrent + attention models.
    model_clip = {
        'GRU':                  0.5,
        'StackedLSTM':          0.5,
        'CNN+BiLSTM+Attention': 0.5,
    }.get(name, 1.0)
    opt = torch.optim.AdamW(model.parameters(), lr=model_lr, weight_decay=1e-4)
    sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=EPOCHS)
    criterion = FocalLoss(alpha=0.25, gamma=2.0)

    best_auc = -1.0
    bad = 0
    history = {'train_loss': [], 'val_auc_mean': []}
    ckpt = MODELS_DIR / f'{name.replace("+","_")}.pt'

    def _safe_sigmoid(L):
        L = np.clip(L, -30.0, 30.0)
        return 1.0 / (1.0 + np.exp(-L))

    for ep in range(1, EPOCHS + 1):
        model.train()
        tot, n, nan_batches = 0.0, 0, 0
        for xb, yb in train_ld:
            xb = xb.to(DEVICE); yb = yb.to(DEVICE)
            if not torch.isfinite(xb).all():
                xb = torch.nan_to_num(xb, nan=0.0, posinf=5.0, neginf=-5.0)
            opt.zero_grad()
            logits = model(xb)
            loss = criterion(logits, yb)
            if not torch.isfinite(loss):
                nan_batches += 1
                continue
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), model_clip)
            opt.step()
            tot += loss.item() * len(xb); n += len(xb)
        sched.step()
        train_loss = tot / max(n, 1)
        if nan_batches:
            print(f'  (skipped {nan_batches} NaN batches)')

        # Validation AUC (mean across horizons), NaN-safe
        model.eval()
        all_logits, all_targets = [], []
        with torch.no_grad():
            for xb, yb in test_ld:
                xb = xb.to(DEVICE)
                if not torch.isfinite(xb).all():
                    xb = torch.nan_to_num(xb, nan=0.0, posinf=5.0, neginf=-5.0)
                logits = model(xb).cpu().numpy()
                all_logits.append(logits); all_targets.append(yb.numpy())
        L = np.concatenate(all_logits); Y = np.concatenate(all_targets)
        L = np.nan_to_num(L, nan=0.0, posinf=30.0, neginf=-30.0)
        P = _safe_sigmoid(L)
        aucs = []
        for j, h in enumerate(HORIZONS):
            yj = Y[:, j]; pj = P[:, j]
            mask = np.isfinite(pj)
            if mask.sum() < 10 or yj[mask].sum() == 0 or yj[mask].sum() == mask.sum():
                aucs.append(float('nan'))
                continue
            aucs.append(roc_auc_score(yj[mask], pj[mask]))
        auc_mean = float(np.nanmean(aucs)) if any(np.isfinite(aucs)) else float('nan')
        history['train_loss'].append(train_loss)
        history['val_auc_mean'].append(auc_mean)
        print(f'  ep{ep:02d}  loss={train_loss:.4f}  AUC[{",".join(f"{a:.3f}" for a in aucs)}]  mean={auc_mean:.4f}')

        if np.isfinite(auc_mean) and auc_mean > best_auc:
            best_auc = auc_mean; bad = 0
            torch.save(model.state_dict(), ckpt)
        else:
            bad += 1
            if bad >= PATIENCE:
                print(f'  early stop @ ep{ep}')
                break

    if not ckpt.exists():
        print(f'  WARNING: no valid checkpoint saved for {name} — using last weights')
        torch.save(model.state_dict(), ckpt)

    # Load best, final eval (NaN-safe)
    model.load_state_dict(torch.load(ckpt))
    model.eval()
    all_logits, all_targets = [], []
    with torch.no_grad():
        for xb, yb in test_ld:
            xb = xb.to(DEVICE)
            if not torch.isfinite(xb).all():
                xb = torch.nan_to_num(xb, nan=0.0, posinf=5.0, neginf=-5.0)
            logits = model(xb).cpu().numpy()
            all_logits.append(logits); all_targets.append(yb.numpy())
    L = np.concatenate(all_logits); Y = np.concatenate(all_targets)
    L = np.nan_to_num(L, nan=0.0, posinf=30.0, neginf=-30.0)
    P = _safe_sigmoid(L)

    summary = {'model': name, 'best_auc_mean': best_auc, 'history': history,
               'per_horizon': {}}
    for j, h in enumerate(HORIZONS):
        yj = Y[:, j]; pj = P[:, j]
        mask = np.isfinite(pj)
        if mask.sum() < 10 or yj[mask].sum() == 0:
            summary['per_horizon'][h] = {'roc_auc': float('nan'), 'pr_auc': float('nan'),
                                         'f1': float('nan'), 'f1_thr05': float('nan'),
                                         'threshold': float('nan')}
            continue
        auc  = roc_auc_score(yj[mask], pj[mask])
        prc  = average_precision_score(yj[mask], pj[mask])
        # F1 at fixed deployment threshold 0.5 (for production realism)
        f1_05 = f1_score(yj[mask], (pj[mask] >= 0.5).astype(int), zero_division=0)
        # F1-optimal threshold sweep (best case)
        ts = np.linspace(0.05, 0.95, 19)
        best_f1, best_t = 0.0, 0.5
        for t in ts:
            f = f1_score(yj[mask], (pj[mask] >= t).astype(int), zero_division=0)
            if f > best_f1:
                best_f1 = f; best_t = float(t)
        summary['per_horizon'][h] = {
            'roc_auc': float(auc), 'pr_auc': float(prc),
            'f1': float(best_f1), 'f1_thr05': float(f1_05),
            'threshold': best_t,
        }
        print(f'   final h={h:>3}s  AUC={auc:.4f}  PR-AUC={prc:.4f}  F1={best_f1:.4f}  F1@.5={f1_05:.4f}  thr={best_t:.2f}')
    return summary

print('Training loop ready (NaN-safe + grad clip).')


Training loop ready (NaN-safe + grad clip).


In [13]:
# ── Cell 11.5: Persistence + XGBoost flat baselines ────────────────────────
# Critical sanity baselines:
#   - Persistence: ŷ_{t+h} = y_t (attack_sec at the last second of the
#     lookback).  Any deep model must clearly beat this to claim it learned
#     temporal patterns rather than just inertia.
#   - XGBoost flat: treats the (T, F) lookback as a flat (T*F,) feature
#     vector — same input as the DL models but no sequence inductive bias.
#     Use to test whether sequence modelling is actually necessary.

import xgboost as xgb

# Build attack_sec for the LAST timestep of each sequence (= y_t).
# We need the binary attack_sec aligned with the test sequences.
# Reconstruct from agg_df using indices we kept in Cell 7.
print('Computing persistence + XGBoost flat baselines on test slice...\n')

# Get y_t (attack at last second of lookback) for train and test
# X_*_seq[:, -1, :] is the last timestep; the binary attack_sec at that step
# is encoded in feature 1 ("attack_flow_count") > 0  →  unscale-aware proxy:
# We instead recompute from y_arr cached in Cell 7.  Easiest: use y_te
# itself at horizon=0?  We did not save horizon=0.  So we'll compute
# persistence from agg_df by re-walking the splits.
#
# Simpler & correct: persistence on test is "y at index t-1 of the sequence"
# which is just the LAST attack_flow_count > 0 of the lookback window.
# That value is in X_te_seq[:, -1, AGG_FEATURE_NAMES.index('attack_flow_count')]
# but it was SCALED.  So we save the *unscaled* persistence flag during Cell 7.
# Quick workaround: derive persistence from y_arr for HORIZON 0 — but we
# only stored y_arr[h] for h in HORIZONS.  Instead, we compute persistence
# on the fly: ŷ_persist = 1 iff *any* of the last L_PERS seconds had attack.

# Build persistence labels from agg_df: y_persist[i] = attack_sec at time t
# of sample i.  Re-walk to recover indices used by build_sequences.
LOOKBACK_LOCAL = LOOKBACK
HORIZONS_LOCAL = HORIZONS
STRIDE_LOCAL   = STRIDE   # references the STRIDE defined in Cell 7

persist_per_sample = []
sample_in_test = []
for src, sub in agg_df.groupby('src_file', sort=False):
    sub = sub.reset_index(drop=True)
    targets_bin = (sub['attack_flow_count'] > 0).astype(np.int64).to_numpy()
    T = len(sub)
    if T < LOOKBACK_LOCAL + max(HORIZONS_LOCAL) + 1:
        continue
    for t in range(LOOKBACK_LOCAL, T - max(HORIZONS_LOCAL), STRIDE_LOCAL):
        persist_per_sample.append(int(targets_bin[t - 1]))
persist_per_sample = np.asarray(persist_per_sample, dtype=np.int64)
assert len(persist_per_sample) == len(X_seq), \
    f'persistence array mismatch: {len(persist_per_sample)} vs {len(X_seq)}'

# Apply same train/test mask
persist_test = persist_per_sample[~mask_train]

# Evaluate persistence at each horizon
persist_summary = {'model': 'Persistence', 'best_auc_mean': None,
                   'history': {'train_loss': [], 'val_auc_mean': []},
                   'per_horizon': {}, 'n_params': 0}
auc_list = []
for h in HORIZONS:
    yj = y_te[h]
    pj = persist_test.astype(np.float32)
    if yj.sum() == 0 or yj.sum() == len(yj):
        persist_summary['per_horizon'][h] = {'roc_auc': float('nan'),
            'pr_auc': float('nan'), 'f1': float('nan'),
            'f1_thr05': float('nan'), 'threshold': float('nan')}
        continue
    auc  = roc_auc_score(yj, pj)
    prc  = average_precision_score(yj, pj)
    f_05 = f1_score(yj, (pj >= 0.5).astype(int), zero_division=0)
    persist_summary['per_horizon'][h] = {'roc_auc': float(auc),
        'pr_auc': float(prc), 'f1': float(f_05),
        'f1_thr05': float(f_05), 'threshold': 0.5}
    auc_list.append(auc)
    print(f'  Persistence  h={h:>3}s  AUC={auc:.4f}  PR-AUC={prc:.4f}  F1={f_05:.4f}')
persist_summary['best_auc_mean'] = float(np.mean(auc_list)) if auc_list else float('nan')

# XGBoost flat baseline
print('\nTraining XGBoost on flattened lookback (T*F features)...')
X_tr_flat = X_tr_seq.reshape(len(X_tr_seq), -1)
X_te_flat = X_te_seq.reshape(len(X_te_seq), -1)
xgb_summary = {'model': 'XGBoost-flat', 'best_auc_mean': None,
               'history': {'train_loss': [], 'val_auc_mean': []},
               'per_horizon': {}}
xgb_aucs = []
t0 = time.time()
for h in HORIZONS:
    yt, yv = y_tr[h], y_te[h]
    if yt.sum() == 0 or yt.sum() == len(yt):
        print(f'  XGB h={h}s SKIP (single-class train)')
        continue
    clf = xgb.XGBClassifier(
        objective='binary:logistic', tree_method='hist',
        device='cuda' if DEVICE.type == 'cuda' else 'cpu',
        max_depth=6, learning_rate=0.1, n_estimators=200,
        subsample=0.8, colsample_bytree=0.8,
        scale_pos_weight=(len(yt) - yt.sum()) / max(yt.sum(), 1),
        random_state=RANDOM_SEED, eval_metric='logloss',
    )
    clf.fit(X_tr_flat, yt, eval_set=[(X_te_flat, yv)], verbose=False)
    proba = clf.predict_proba(X_te_flat)[:, 1]
    auc = roc_auc_score(yv, proba)
    prc = average_precision_score(yv, proba)
    f_05 = f1_score(yv, (proba >= 0.5).astype(int), zero_division=0)
    # F1-optimal threshold sweep
    best_f1, best_t = 0.0, 0.5
    for t in np.linspace(0.05, 0.95, 19):
        f = f1_score(yv, (proba >= t).astype(int), zero_division=0)
        if f > best_f1: best_f1, best_t = f, float(t)
    xgb_summary['per_horizon'][h] = {'roc_auc': float(auc), 'pr_auc': float(prc),
        'f1': float(best_f1), 'f1_thr05': float(f_05), 'threshold': best_t}
    xgb_aucs.append(auc)
    print(f'  XGB-flat     h={h:>3}s  AUC={auc:.4f}  PR-AUC={prc:.4f}  F1={best_f1:.4f}  F1@.5={f_05:.4f}  thr={best_t:.2f}')

xgb_summary['best_auc_mean'] = float(np.mean(xgb_aucs)) if xgb_aucs else float('nan')
xgb_summary['n_params'] = int(X_tr_flat.shape[1])  # # input features
xgb_summary['fit_sec'] = round(time.time() - t0, 1)
print(f'\nXGBoost-flat done in {xgb_summary["fit_sec"]}s,  flat feature dim = {xgb_summary["n_params"]}')

baselines_extra = [persist_summary, xgb_summary]


Computing persistence + XGBoost flat baselines on test slice...

  Persistence  h= 60s  AUC=0.8481  PR-AUC=0.6466  F1=0.7689
  Persistence  h=300s  AUC=0.8555  PR-AUC=0.6566  F1=0.7776
  Persistence  h=600s  AUC=0.8627  PR-AUC=0.6505  F1=0.7776

Training XGBoost on flattened lookback (T*F features)...
  XGB-flat     h= 60s  AUC=0.9525  PR-AUC=0.7729  F1=0.8724  F1@.5=0.8715  thr=0.55
  XGB-flat     h=300s  AUC=0.9452  PR-AUC=0.7389  F1=0.8494  F1@.5=0.8474  thr=0.65
  XGB-flat     h=600s  AUC=0.9322  PR-AUC=0.6575  F1=0.8283  F1@.5=0.7851  thr=0.25

XGBoost-flat done in 83.5s,  flat feature dim = 10800


In [14]:
# ── Cell 12: Train all forecasters ─────────────────────────────────────────
summaries = list(baselines_extra)   # Persistence + XGBoost-flat already done
stored_probas = {}                  # for ensemble in next cell
for name, Klass in MODEL_FACTORY.items():
    # Reset all random sources for reproducible per-model init
    torch.manual_seed(RANDOM_SEED)
    np.random.seed(RANDOM_SEED)
    torch.cuda.manual_seed_all(RANDOM_SEED)
    model = Klass()
    n_params = sum(p.numel() for p in model.parameters())
    print(f'[{name}] params={n_params/1e3:.1f}K')
    s = train_model(name, model)
    s['n_params'] = n_params
    summaries.append(s)
    # Save probabilities for ensemble (re-derived from saved checkpoint)
    ckpt = MODELS_DIR / f'{name.replace("+","_")}.pt'
    if ckpt.exists():
        model.load_state_dict(torch.load(ckpt))
        model.eval()
        probas = []
        with torch.no_grad():
            for xb, _ in test_ld:
                xb = xb.to(DEVICE)
                if not torch.isfinite(xb).all():
                    xb = torch.nan_to_num(xb, nan=0.0, posinf=5.0, neginf=-5.0)
                p = torch.sigmoid(torch.clamp(model(xb), -30, 30)).cpu().numpy()
                probas.append(p)
        stored_probas[name] = np.concatenate(probas)

with open(RESULT_DIR / 'forecast_summaries.json', 'w') as f:
    json.dump(summaries, f, indent=2)


[StackedLSTM] params=348.4K

=== Training StackedLSTM ===
  ep01  loss=0.0551  AUC[0.699,0.684,0.663]  mean=0.6818
  ep02  loss=0.0246  AUC[0.727,0.712,0.687]  mean=0.7085
  ep03  loss=0.0233  AUC[0.777,0.759,0.733]  mean=0.7562
  ep04  loss=0.0226  AUC[0.777,0.761,0.736]  mean=0.7581
  ep05  loss=0.0219  AUC[0.799,0.779,0.753]  mean=0.7770
  ep06  loss=0.0206  AUC[0.855,0.834,0.811]  mean=0.8334
  ep07  loss=0.0199  AUC[0.857,0.838,0.814]  mean=0.8364
  ep08  loss=0.0189  AUC[0.919,0.907,0.885]  mean=0.9039
  ep09  loss=0.0215  AUC[0.817,0.787,0.762]  mean=0.7885
  ep10  loss=0.0207  AUC[0.844,0.836,0.812]  mean=0.8305
  ep11  loss=0.0189  AUC[0.803,0.801,0.772]  mean=0.7920
  ep12  loss=0.0179  AUC[0.886,0.886,0.862]  mean=0.8780
  early stop @ ep12
   final h= 60s  AUC=0.9195  PR-AUC=0.7358  F1=0.7298  F1@.5=0.3346  thr=0.20
   final h=300s  AUC=0.9069  PR-AUC=0.6873  F1=0.7253  F1@.5=0.1760  thr=0.20
   final h=600s  AUC=0.8854  PR-AUC=0.5765  F1=0.6394  F1@.5=0.1000  thr=0.20
[GRU

---
## Section 4 — Comparison plots

In [16]:
# ── Cell 13: Tabulate per-horizon scores ────────────────────────────────────
rows = []
for s in summaries:
    for h, m in s['per_horizon'].items():
        rows.append({'model': s['model'], 'horizon_s': int(h),
                     'roc_auc':   m.get('roc_auc',   float('nan')),
                     'pr_auc':    m.get('pr_auc',    float('nan')),
                     'f1':        m.get('f1',        float('nan')),
                     'f1_thr05':  m.get('f1_thr05',  float('nan')),
                     'threshold': m.get('threshold', float('nan'))})
df_fc = pd.DataFrame(rows)
print('=== Per-horizon scores (incl. F1 at fixed thr=0.5 for deployment realism) ===\n')
print(df_fc.pivot(index='model', columns='horizon_s',
                  values=['roc_auc', 'pr_auc', 'f1', 'f1_thr05']).round(4))
df_fc.to_csv(RESULT_DIR / 'forecast_metrics.csv', index=False)

# Lift over Persistence baseline (Δ-AUC per model per horizon)
persist_aucs = {}
for r in rows:
    if r['model'] == 'Persistence':
        persist_aucs[r['horizon_s']] = r['roc_auc']
if persist_aucs:
    print('\n=== Lift over Persistence (ΔAUC) ===')
    lift_rows = []
    for r in rows:
        if r['model'] == 'Persistence':
            continue
        base = persist_aucs.get(r['horizon_s'], float('nan'))
        lift_rows.append({'model': r['model'], 'horizon_s': r['horizon_s'],
                          'auc_model': r['roc_auc'], 'auc_persistence': base,
                          'delta_auc': r['roc_auc'] - base})
    df_lift = pd.DataFrame(lift_rows)
    print(df_lift.pivot(index='model', columns='horizon_s', values='delta_auc').round(4))
    df_lift.to_csv(RESULT_DIR / 'lift_over_persistence.csv', index=False)


=== Per-horizon scores (incl. F1 at fixed thr=0.5 for deployment realism) ===

                     roc_auc                  pr_auc                      f1  \
horizon_s                60      300     600     60      300     600     60    
model                                                                          
CNN+BiLSTM+Attention  0.9445  0.9540  0.9593  0.7576  0.7878  0.7977  0.8302   
GRU                   0.8197  0.7921  0.7984  0.5266  0.4836  0.4265  0.3960   
Persistence           0.8481  0.8555  0.8627  0.6466  0.6566  0.6505  0.7689   
StackedLSTM           0.9195  0.9069  0.8854  0.7358  0.6873  0.5765  0.7298   
TemporalCNN           0.9491  0.9476  0.9162  0.7649  0.7653  0.6610  0.8302   
Transformer+LSTM      0.9368  0.9338  0.9177  0.7498  0.7166  0.6184  0.6731   
XGBoost-flat          0.9525  0.9452  0.9322  0.7729  0.7389  0.6575  0.8724   

                                     f1_thr05                  
horizon_s                300     600      60      300   

In [17]:
# ── Cell 14: Plot ROC-AUC per horizon, per model ────────────────────────────
fig, ax = plt.subplots(figsize=(8, 4.5))
for s in summaries:
    xs = HORIZONS
    ys = [s['per_horizon'][h]['roc_auc'] for h in HORIZONS]
    ax.plot(xs, ys, marker='o', label=s['model'])
ax.set_xlabel('Forecast horizon (seconds)')
ax.set_ylabel('ROC-AUC')
ax.set_title('CSE-CIC-IDS2018 — attack forecast ROC-AUC by horizon')
ax.set_xticks(HORIZONS)
ax.set_ylim(0.5, 1.0)
ax.grid(alpha=0.3)
ax.legend()
plt.tight_layout()
plt.savefig(FIG_DIR / 'forecast_auc_vs_horizon.png', dpi=140)
plt.close()

# F1
fig, ax = plt.subplots(figsize=(8, 4.5))
for s in summaries:
    ys = [s['per_horizon'][h]['f1'] for h in HORIZONS]
    ax.plot(HORIZONS, ys, marker='s', label=s['model'])
ax.set_xlabel('Forecast horizon (seconds)')
ax.set_ylabel('Best-threshold F1')
ax.set_title('CSE-CIC-IDS2018 — attack forecast F1 by horizon')
ax.set_xticks(HORIZONS)
ax.grid(alpha=0.3)
ax.legend()
plt.tight_layout()
plt.savefig(FIG_DIR / 'forecast_f1_vs_horizon.png', dpi=140)
plt.close()
print('Saved figures to', FIG_DIR)


Saved figures to /kaggle/working/pad_onap_forecast/figures


In [18]:
# ── Cell 15: Training-curve plot for each model ─────────────────────────────
for s in summaries:
    fig, axes = plt.subplots(1, 2, figsize=(11, 3.6))
    axes[0].plot(s['history']['train_loss'], color='#4C78A8')
    axes[0].set_title(f'{s["model"]} — train loss')
    axes[0].set_xlabel('epoch')
    axes[1].plot(s['history']['val_auc_mean'], color='#E45756')
    axes[1].set_title(f'{s["model"]} — val mean AUC')
    axes[1].set_xlabel('epoch'); axes[1].set_ylim(0.5, 1.0)
    for a in axes:
        a.grid(alpha=0.3)
    plt.tight_layout()
    plt.savefig(FIG_DIR / f'curves_{s["model"].replace("+","_")}.png', dpi=140)
    plt.close()
print('Done.')


Done.


In [19]:
# ── Cell 16: Package outputs ────────────────────────────────────────────────
import shutil
zip_path = shutil.make_archive('/kaggle/working/pad_onap_forecast', 'zip', OUTPUT_DIR)
print('Archive ->', zip_path)
print('\nForecast summary (mean ROC-AUC across horizons):')
ranking = sorted(summaries, key=lambda s: -s['best_auc_mean'])
for s in ranking:
    print(f'  {s["model"]:<20}  {s["best_auc_mean"]:.4f}  ({s["n_params"]/1e3:.1f}K params)')


Archive -> /kaggle/working/pad_onap_forecast.zip

Forecast summary (mean ROC-AUC across horizons):
  CNN+BiLSTM+Attention  0.9526  (190.6K params)
  XGBoost-flat          0.9433  (10.8K params)
  TemporalCNN           0.9377  (41.3K params)
  Transformer+LSTM      0.9294  (540.0K params)
  StackedLSTM           0.9039  (348.4K params)
  Persistence           0.8554  (0.0K params)
  GRU                   0.8034  (263.4K params)
